# Model Notebook

**Financial Tweet Sentiment Classification — Nova IMS Text Mining 2025/2026**

End-to-end pipeline: majority baseline → preprocessing → TF-IDF → classical ML → word embeddings → error analysis → DistilBERT fine-tuning.

Run top-to-bottom. Leaderboard logging is idempotent — re-running the same experiment overwrites rather than duplicates.

In [1]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Working directory: {os.getcwd()}")


Working directory: c:\Users\MartaFeria\Documents\NovaIms\TextMining\Project\repo


In [2]:
%pip install -q -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

from src.config import TRAIN_CSV_PATH, TEST_CSV_PATH, RESULTS_CSV_PATH, SEED, LABEL_NAMES
from src.preprocessing import stratified_split, preprocess_tweet, run_smoke_tests
from src.evaluate import evaluate_and_log
from src.experiment import run_tfidf_pipeline, run_model_pipeline
from src.word_embeddings import calculate_oov, vectorize_corpus, train_word2vec, load_glove_twitter

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110})
print("Imports OK")


Imports OK


## 1. Load Data & Split

In [4]:
train_df = pd.read_csv(TRAIN_CSV_PATH)
print(f"Total training examples: {len(train_df)}")
print(train_df["label"].value_counts().sort_index().rename(LABEL_NAMES))


Total training examples: 9543
label
Bearish    1442
Bullish    1923
Neutral    6178
Name: count, dtype: int64


In [5]:
X_train_raw, X_val_raw, y_train, y_val = stratified_split(train_df)
print(f"Train: {len(X_train_raw)} | Val: {len(X_val_raw)}")


Train: 7634 | Val: 1909


## 2. Majority-Class Baseline

In [6]:
from src.experiment import run_majority_baseline
run_majority_baseline(y_train, y_val)


MODEL EVALUATION: Majority Class Baseline (N/A)
[INFO] Accuracy          : 0.6475
[INFO] Precision (Macro) : 0.2158
[INFO] Recall (Macro)    : 0.3333
[INFO] F1 (Macro)        : 0.2620
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.00      0.00      0.00       288
     Bullish       0.00      0.00      0.00       385
     Neutral       0.65      1.00      0.79      1236

    accuracy                           0.65      1909
   macro avg       0.22      0.33      0.26      1909
weighted avg       0.42      0.65      0.51      1909

------------------------------------------------------------
[[   0    0  288]
 [   0    0  385]
 [   0    0 1236]]
[INFO] Updated run in outputs/results.csv


{'accuracy': 0.6474594028287062,
 'precision_macro': 0.21581980094290207,
 'recall_macro': 0.3333333333333333,
 'f1_macro': 0.2620031796502385,
 'f1_per_class': {'Bearish': 0.0,
  'Bullish': 0.0,
  'Neutral': 0.7860095389507155}}

## 3. Preprocessing Smoke Test

In [7]:
run_smoke_tests()


PREPROCESSING SMOKE TESTS
[INFO] Test 1: 'Downgrades 4/7: $MLND to underperform at Needham—see details... https://t.co/example'
[INFO]   Lemmatized : 'downgrade 4/7 : $ mlnd underperform needham-see detail ... URL_PLACEHOLDER'
[INFO]   Stemmed    : ['downgrad', '4/7', ':', 'CASHTAG_PLACEHOLDER', 'underperform', 'needham-se', 'detail', '...']
[INFO] Test 2: 'Shorting $AAPL here at $180. @elonmusk thoughts? #market #trading'
[INFO]   Lemmatized : 'shorting $ aapl $ 180 . MENTION_PLACEHOLDER thought ? #market #trading'
[INFO]   Stemmed    : ['short', 'CASHTAG_PLACEHOLDER', 'CASHTAG_PLACEHOLDER', '.', 'thought', '?', '#market', '#trade']
[INFO] Test 3: 'RT @NovaIMS: $BTC is falling down rapidly... RT to warn others!'
[INFO]   Lemmatized : 'MENTION_PLACEHOLDER : $ btc falling rapidly ... warn others !'
[INFO]   Stemmed    : [':', 'CASHTAG_PLACEHOLDER', 'fall', 'rapidli', '...', 'warn', 'other', '!']
[INFO] Test 4: 'Having a cup of coffee and watching the market open. Very neutral.'
[INFO]  

## 4. Preprocess Corpus

In [8]:
print("Preprocessing training set...")
X_train_pre = X_train_raw.apply(lambda t: preprocess_tweet(t, return_str=True))
X_val_pre   = X_val_raw.apply(lambda t: preprocess_tweet(t, return_str=True))
print(f"Sample preprocessed tweet:\n  raw: {X_train_raw.iloc[0]}\n  pre: {X_train_pre.iloc[0]}")


Preprocessing training set...
Sample preprocessed tweet:
  raw: Nasdaq prices 600M of 0.875% senior notes
  pre: nasdaq price 600m 0.875 % senior note


## 5. TF-IDF Feature Extraction Experiments

### 5A — TF-IDF Unigrams Baseline

In [9]:
vocab_uni, metrics_uni = run_tfidf_pipeline(
    X_train_pre, X_val_pre, y_train, y_val,
    ngram_range=(1, 1), min_df=1, max_features=None,
    feature_desc="TF-IDF (1,1)"
)
print(f"Vocabulary size: {vocab_uni:,}  |  Macro F1: {metrics_uni['f1_macro']:.4f}")


MODEL EVALUATION: Logistic Regression Baseline (TF-IDF (1,1))
[INFO] Accuracy          : 0.7685
[INFO] Precision (Macro) : 0.6910
[INFO] Recall (Macro)    : 0.7015
[INFO] F1 (Macro)        : 0.6955
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.54      0.61      0.58       288
     Bullish       0.67      0.65      0.66       385
     Neutral       0.86      0.84      0.85      1236

    accuracy                           0.77      1909
   macro avg       0.69      0.70      0.70      1909
weighted avg       0.77      0.77      0.77      1909

------------------------------------------------------------
[[ 176   40   72]
 [  35  251   99]
 [ 113   83 1040]]
[INFO] Updated run in outputs/results.csv
Vocabulary size: 12,575  |  Macro F1: 0.6955


### 5B — TF-IDF Unigrams + Bigrams (Raw)

In [10]:
vocab_bi, metrics_bi = run_tfidf_pipeline(
    X_train_pre, X_val_pre, y_train, y_val,
    ngram_range=(1, 2), min_df=1, max_features=None,
    feature_desc="TF-IDF (1,2)"
)
print(f"Vocabulary size: {vocab_bi:,}  |  Macro F1: {metrics_bi['f1_macro']:.4f}")


MODEL EVALUATION: Logistic Regression Baseline (TF-IDF (1,2))
[INFO] Accuracy          : 0.7800
[INFO] Precision (Macro) : 0.7019
[INFO] Recall (Macro)    : 0.7057
[INFO] F1 (Macro)        : 0.7038
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.58      0.59      0.59       288
     Bullish       0.66      0.67      0.66       385
     Neutral       0.87      0.86      0.86      1236

    accuracy                           0.78      1909
   macro avg       0.70      0.71      0.70      1909
weighted avg       0.78      0.78      0.78      1909

------------------------------------------------------------
[[ 170   47   71]
 [  35  257   93]
 [  87   87 1062]]
[INFO] Updated run in outputs/results.csv
Vocabulary size: 58,149  |  Macro F1: 0.7038


### 5C — TF-IDF Unigrams + Bigrams (Optimized, min_df=2, max_features=25k)

In [11]:
vocab_opt, metrics_opt = run_tfidf_pipeline(
    X_train_pre, X_val_pre, y_train, y_val,
    ngram_range=(1, 2), min_df=2, max_features=25000,
    feature_desc="TF-IDF (1,2) Optimized"
)
print(f"Vocabulary size: {vocab_opt:,}  |  Macro F1: {metrics_opt['f1_macro']:.4f}")


MODEL EVALUATION: Logistic Regression Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7795
[INFO] Precision (Macro) : 0.7021
[INFO] Recall (Macro)    : 0.7175
[INFO] F1 (Macro)        : 0.7091
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.56      0.64      0.60       288
     Bullish       0.67      0.67      0.67       385
     Neutral       0.87      0.85      0.86      1236

    accuracy                           0.78      1909
   macro avg       0.70      0.72      0.71      1909
weighted avg       0.79      0.78      0.78      1909

------------------------------------------------------------
[[ 183   43   62]
 [  37  258   90]
 [ 104   85 1047]]
[INFO] Updated run in outputs/results.csv
Vocabulary size: 11,018  |  Macro F1: 0.7091


## 6. Classical ML Model Comparison

All models use the Optimized TF-IDF feature space.

In [12]:
# Build shared optimised TF-IDF matrix once
vec_opt = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000)
X_train_opt = vec_opt.fit_transform(X_train_pre)
X_val_opt   = vec_opt.transform(X_val_pre)
print(f"Feature matrix shape: train={X_train_opt.shape} | val={X_val_opt.shape}")


Feature matrix shape: train=(7634, 11018) | val=(1909, 11018)


### 6A — K-Nearest Neighbours

In [13]:
metrics_knn3 = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=KNeighborsClassifier(n_neighbors=3),
    model_name="KNN Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="n_neighbors=3"
)


MODEL EVALUATION: KNN Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.6888
[INFO] Precision (Macro) : 0.7305
[INFO] Recall (Macro)    : 0.4212
[INFO] F1 (Macro)        : 0.4247
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.68      0.12      0.21       288
     Bullish       0.83      0.15      0.25       385
     Neutral       0.68      0.99      0.81      1236

    accuracy                           0.69      1909
   macro avg       0.73      0.42      0.42      1909
weighted avg       0.71      0.69      0.61      1909

------------------------------------------------------------
[[  36    6  246]
 [   8   58  319]
 [   9    6 1221]]
[INFO] Updated run in outputs/results.csv


In [14]:
metrics_knn7 = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=KNeighborsClassifier(n_neighbors=7),
    model_name="KNN Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="n_neighbors=7"
)


MODEL EVALUATION: KNN Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.6695
[INFO] Precision (Macro) : 0.8276
[INFO] Recall (Macro)    : 0.3759
[INFO] F1 (Macro)        : 0.3463
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.88      0.05      0.10       288
     Bullish       0.94      0.08      0.14       385
     Neutral       0.66      1.00      0.80      1236

    accuracy                           0.67      1909
   macro avg       0.83      0.38      0.35      1909
weighted avg       0.75      0.67      0.56      1909

------------------------------------------------------------
[[  15    0  273]
 [   1   30  354]
 [   1    2 1233]]
[INFO] Updated run in outputs/results.csv


In [15]:
knn_cv = GridSearchCV(
    KNeighborsClassifier(),
    param_grid={"n_neighbors": [3, 7]},
    cv=3, scoring="f1_macro", n_jobs=-1
)
knn_cv.fit(X_train_opt, y_train)
best_k = knn_cv.best_params_["n_neighbors"]
print(f"Best n_neighbors: {best_k}")

from src.evaluate import evaluate_and_log
metrics_knn_best = evaluate_and_log(
    y_val, knn_cv.predict(X_val_opt),
    model_name="KNN Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params=f"GridSearchCV Best, n_neighbors={best_k}"
)


Best n_neighbors: 3
MODEL EVALUATION: KNN Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.6888
[INFO] Precision (Macro) : 0.7305
[INFO] Recall (Macro)    : 0.4212
[INFO] F1 (Macro)        : 0.4247
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.68      0.12      0.21       288
     Bullish       0.83      0.15      0.25       385
     Neutral       0.68      0.99      0.81      1236

    accuracy                           0.69      1909
   macro avg       0.73      0.42      0.42      1909
weighted avg       0.71      0.69      0.61      1909

------------------------------------------------------------
[[  36    6  246]
 [   8   58  319]
 [   9    6 1221]]
[INFO] Updated run in outputs/results.csv


### 6B — Logistic Regression Variants

In [16]:
metrics_lr_l1 = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=LogisticRegression(penalty="l1", solver="saga", max_iter=1000,
                             class_weight="balanced", random_state=SEED),
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="penalty=l1, solver=saga, C=1.0, class_weight=balanced"
)


MODEL EVALUATION: Logistic Regression Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7454
[INFO] Precision (Macro) : 0.6611
[INFO] Recall (Macro)    : 0.6941
[INFO] F1 (Macro)        : 0.6745
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.48      0.61      0.54       288
     Bullish       0.63      0.68      0.65       385
     Neutral       0.87      0.80      0.83      1236

    accuracy                           0.75      1909
   macro avg       0.66      0.69      0.67      1909
weighted avg       0.76      0.75      0.75      1909

------------------------------------------------------------
[[175  53  60]
 [ 40 260  85]
 [146 102 988]]
[INFO] Updated run in outputs/results.csv


In [17]:
metrics_lr_l2 = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000,
                             class_weight="balanced", random_state=SEED),
    model_name="Logistic Regression Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="penalty=l2, solver=lbfgs, C=1.0, class_weight=balanced"
)


MODEL EVALUATION: Logistic Regression Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7795
[INFO] Precision (Macro) : 0.7021
[INFO] Recall (Macro)    : 0.7175
[INFO] F1 (Macro)        : 0.7091
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.56      0.64      0.60       288
     Bullish       0.67      0.67      0.67       385
     Neutral       0.87      0.85      0.86      1236

    accuracy                           0.78      1909
   macro avg       0.70      0.72      0.71      1909
weighted avg       0.79      0.78      0.78      1909

------------------------------------------------------------
[[ 183   43   62]
 [  37  258   90]
 [ 104   85 1047]]
[INFO] Updated run in outputs/results.csv


### 6C — Multinomial Naive Bayes

In [18]:
metrics_nb1 = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=MultinomialNB(alpha=1.0),
    model_name="Multinomial NB Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="alpha=1.0"
)


MODEL EVALUATION: Multinomial NB Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7339
[INFO] Precision (Macro) : 0.7944
[INFO] Recall (Macro)    : 0.5000
[INFO] F1 (Macro)        : 0.5316
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.85      0.16      0.26       288
     Bullish       0.81      0.36      0.50       385
     Neutral       0.72      0.99      0.83      1236

    accuracy                           0.73      1909
   macro avg       0.79      0.50      0.53      1909
weighted avg       0.76      0.73      0.68      1909

------------------------------------------------------------
[[  45   19  224]
 [   3  138  244]
 [   5   13 1218]]
[INFO] Updated run in outputs/results.csv


In [19]:
metrics_nb01 = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=MultinomialNB(alpha=0.1),
    model_name="Multinomial NB Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="alpha=0.1"
)


MODEL EVALUATION: Multinomial NB Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7863
[INFO] Precision (Macro) : 0.7527
[INFO] Recall (Macro)    : 0.6415
[INFO] F1 (Macro)        : 0.6791
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.72      0.43      0.54       288
     Bullish       0.74      0.55      0.63       385
     Neutral       0.80      0.94      0.87      1236

    accuracy                           0.79      1909
   macro avg       0.75      0.64      0.68      1909
weighted avg       0.78      0.79      0.77      1909

------------------------------------------------------------
[[ 125   36  127]
 [  18  211  156]
 [  31   40 1165]]
[INFO] Updated run in outputs/results.csv


### 6D — MLP Neural Network

In [20]:
metrics_mlp_s = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=MLPClassifier(hidden_layer_sizes=(100,), learning_rate_init=0.01,
                        early_stopping=True, random_state=SEED),
    model_name="MLP Classifier",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="hidden_layer_sizes=(100,), learning_rate_init=0.01, early_stopping=True"
)


MODEL EVALUATION: MLP Classifier (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7842
[INFO] Precision (Macro) : 0.7203
[INFO] Recall (Macro)    : 0.6707
[INFO] F1 (Macro)        : 0.6914
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.61      0.52      0.56       288
     Bullish       0.72      0.59      0.65       385
     Neutral       0.83      0.91      0.87      1236

    accuracy                           0.78      1909
   macro avg       0.72      0.67      0.69      1909
weighted avg       0.77      0.78      0.78      1909

------------------------------------------------------------
[[ 149   35  104]
 [  34  226  125]
 [  61   53 1122]]
[INFO] Updated run in outputs/results.csv


In [21]:
metrics_mlp_d = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=MLPClassifier(hidden_layer_sizes=(100, 50), learning_rate_init=0.001,
                        early_stopping=True, random_state=SEED),
    model_name="MLP Classifier",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="hidden_layer_sizes=(100, 50), learning_rate_init=0.001, early_stopping=True"
)


MODEL EVALUATION: MLP Classifier (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.8025
[INFO] Precision (Macro) : 0.7590
[INFO] Recall (Macro)    : 0.6728
[INFO] F1 (Macro)        : 0.7051
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.68      0.47      0.56       288
     Bullish       0.77      0.61      0.68       385
     Neutral       0.83      0.94      0.88      1236

    accuracy                           0.80      1909
   macro avg       0.76      0.67      0.71      1909
weighted avg       0.79      0.80      0.79      1909

------------------------------------------------------------
[[ 136   38  114]
 [  23  233  129]
 [  40   33 1163]]
[INFO] Updated run in outputs/results.csv


### 6E — Random Forest

In [22]:
metrics_rf_s = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=RandomForestClassifier(n_estimators=100, max_depth=10,
                                 class_weight="balanced", random_state=SEED, n_jobs=-1),
    model_name="Random Forest Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="n_estimators=100, max_depth=10, class_weight=balanced"
)


MODEL EVALUATION: Random Forest Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7192
[INFO] Precision (Macro) : 0.6334
[INFO] Recall (Macro)    : 0.6208
[INFO] F1 (Macro)        : 0.6235
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.48      0.55      0.51       288
     Bullish       0.61      0.48      0.54       385
     Neutral       0.81      0.83      0.82      1236

    accuracy                           0.72      1909
   macro avg       0.63      0.62      0.62      1909
weighted avg       0.72      0.72      0.72      1909

------------------------------------------------------------
[[ 158   39   91]
 [  42  185  158]
 [ 129   77 1030]]
[INFO] Updated run in outputs/results.csv


In [23]:
metrics_rf_d = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=RandomForestClassifier(n_estimators=200, max_depth=20,
                                 class_weight="balanced", random_state=SEED, n_jobs=-1),
    model_name="Random Forest Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="n_estimators=200, max_depth=20, class_weight=balanced"
)


MODEL EVALUATION: Random Forest Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7402
[INFO] Precision (Macro) : 0.6585
[INFO] Recall (Macro)    : 0.6287
[INFO] F1 (Macro)        : 0.6412
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.53      0.50      0.51       288
     Bullish       0.64      0.52      0.57       385
     Neutral       0.81      0.86      0.84      1236

    accuracy                           0.74      1909
   macro avg       0.66      0.63      0.64      1909
weighted avg       0.73      0.74      0.73      1909

------------------------------------------------------------
[[ 144   47   97]
 [  27  201  157]
 [ 101   67 1068]]
[INFO] Updated run in outputs/results.csv


### 6F — XGBoost

In [24]:
metrics_xgb_s = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=XGBClassifier(max_depth=3, learning_rate=0.1, n_estimators=150,
                        random_state=SEED, n_jobs=-1),
    model_name="XGBoost Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="max_depth=3, learning_rate=0.1, n_estimators=150"
)


MODEL EVALUATION: XGBoost Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7349
[INFO] Precision (Macro) : 0.7926
[INFO] Recall (Macro)    : 0.5100
[INFO] F1 (Macro)        : 0.5500
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.84      0.23      0.36       288
     Bullish       0.81      0.32      0.46       385
     Neutral       0.72      0.98      0.83      1236

    accuracy                           0.73      1909
   macro avg       0.79      0.51      0.55      1909
weighted avg       0.76      0.73      0.69      1909

------------------------------------------------------------
[[  65   14  209]
 [   5  124  256]
 [   7   15 1214]]
[INFO] Updated run in outputs/results.csv


In [25]:
metrics_xgb_d = run_model_pipeline(
    X_train_opt, X_val_opt, y_train, y_val,
    model=XGBClassifier(max_depth=6, learning_rate=0.05, n_estimators=200,
                        random_state=SEED, n_jobs=-1),
    model_name="XGBoost Baseline",
    feature_desc="TF-IDF (1,2) Optimized",
    params_str="max_depth=6, learning_rate=0.05, n_estimators=200"
)


MODEL EVALUATION: XGBoost Baseline (TF-IDF (1,2) Optimized)
[INFO] Accuracy          : 0.7381
[INFO] Precision (Macro) : 0.7638
[INFO] Recall (Macro)    : 0.5250
[INFO] F1 (Macro)        : 0.5658
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.80      0.24      0.37       288
     Bullish       0.76      0.36      0.49       385
     Neutral       0.73      0.97      0.83      1236

    accuracy                           0.74      1909
   macro avg       0.76      0.52      0.57      1909
weighted avg       0.75      0.74      0.70      1909

------------------------------------------------------------
[[  70   18  200]
 [   7  139  239]
 [  11   25 1200]]
[INFO] Updated run in outputs/results.csv


## 7. Classical ML Leaderboard Summary

In [26]:
leaderboard = [
    ("TF-IDF (1,1) + LR",          metrics_uni),
    ("TF-IDF (1,2) Raw + LR",       metrics_bi),
    ("TF-IDF (1,2) Opt + LR",       metrics_opt),
    ("KNN k=3",                     metrics_knn3),
    ("KNN k=7",                     metrics_knn7),
    (f"KNN GridCV (k={best_k})",    metrics_knn_best),
    ("LR L1 (Lasso)",               metrics_lr_l1),
    ("LR L2 (Ridge)",               metrics_lr_l2),
    ("NB alpha=1.0",                metrics_nb1),
    ("NB alpha=0.1",                metrics_nb01),
    ("MLP (100,)",                  metrics_mlp_s),
    ("MLP (100, 50)",               metrics_mlp_d),
    ("RF depth=10",                 metrics_rf_s),
    ("RF depth=20",                 metrics_rf_d),
    ("XGB depth=3 lr=0.1",          metrics_xgb_s),
    ("XGB depth=6 lr=0.05",         metrics_xgb_d),
]

lb_df = pd.DataFrame([
    {"Model": name, "Accuracy": m["accuracy"],
     "Precision": m["precision_macro"], "Recall": m["recall_macro"], "F1 Macro": m["f1_macro"]}
    for name, m in leaderboard
]).sort_values("F1 Macro", ascending=False).reset_index(drop=True)

lb_df.index += 1
display(lb_df.style.background_gradient(cmap="YlGn", subset=["F1 Macro"]).format("{:.4f}", subset=lb_df.columns[1:]))


,Model,Accuracy,Precision,Recall,F1 Macro
1,"TF-IDF (1,2) Opt + LR",0.7795,0.7021,0.7175,0.7091
2,LR L2 (Ridge),0.7795,0.7021,0.7175,0.7091
3,"MLP (100, 50)",0.8025,0.7590,0.6728,0.7051
4,"TF-IDF (1,2) Raw + LR",0.7800,0.7019,0.7057,0.7038
5,"TF-IDF (1,1) + LR",0.7685,0.6910,0.7015,0.6955
6,"MLP (100,)",0.7842,0.7203,0.6707,0.6914
7,NB alpha=0.1,0.7863,0.7527,0.6415,0.6791
8,LR L1 (Lasso),0.7454,0.6611,0.6941,0.6745
9,RF depth=20,0.7402,0.6585,0.6287,0.6412
10,RF depth=10,0.7192,0.6334,0.6208,0.6235


## 8. Word Embeddings — Word2Vec vs GloVe-Twitter-100

In [27]:
print("Tokenising corpus for word embeddings (no return_str)...")
X_train_tokens = X_train_raw.apply(lambda t: preprocess_tweet(t, return_str=False)).tolist()
X_val_tokens   = X_val_raw.apply(lambda t: preprocess_tweet(t, return_str=False)).tolist()
print(f"Sample token list: {X_train_tokens[0][:8]}")


Tokenising corpus for word embeddings (no return_str)...
Sample token list: ['nasdaq', 'price', '600m', '0.875', '%', 'senior', 'note']


In [28]:
w2v_model = train_word2vec(X_train_tokens, vector_size=100, window=5, min_count=2)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv.key_to_index):,}")


[SUCCESS] Word2Vec trained — vocab size: 6078
Word2Vec vocabulary size: 6,078


In [29]:
glove_model = load_glove_twitter(dim=100)
print(f"GloVe vocabulary size: {len(glove_model.key_to_index):,}")


[INFO] Loading glove-twitter-100 (downloads if not cached) ...
[SUCCESS] GloVe-Twitter-100 loaded — vocab size: 1193514
GloVe vocabulary size: 1,193,514


In [30]:
w2v_u_oov, w2v_t_oov, _   = calculate_oov(X_val_tokens, w2v_model)
glove_u_oov, glove_t_oov, _ = calculate_oov(X_val_tokens, glove_model)

oov_df = pd.DataFrame({
    "Embedding": ["Custom Word2Vec", "GloVe-Twitter-100"],
    "Unique Word OOV %": [w2v_u_oov * 100, glove_u_oov * 100],
    "Token OOV %": [w2v_t_oov * 100, glove_t_oov * 100],
})
print(oov_df.to_string(index=False))


        Embedding  Unique Word OOV %  Token OOV %
  Custom Word2Vec          45.796125    13.993952
GloVe-Twitter-100          22.510531    17.766923


In [31]:
X_train_w2v   = vectorize_corpus(X_train_tokens, w2v_model, 100)
X_val_w2v     = vectorize_corpus(X_val_tokens,   w2v_model, 100)
X_train_glove = vectorize_corpus(X_train_tokens, glove_model, 100)
X_val_glove   = vectorize_corpus(X_val_tokens,   glove_model, 100)

print(f"W2V matrix shapes  — train: {X_train_w2v.shape} | val: {X_val_w2v.shape}")
print(f"GloVe matrix shapes — train: {X_train_glove.shape} | val: {X_val_glove.shape}")


W2V matrix shapes  — train: (7634, 100) | val: (1909, 100)
GloVe matrix shapes — train: (7634, 100) | val: (1909, 100)


In [32]:
clf_w2v = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000,
                              class_weight="balanced", random_state=SEED)
clf_w2v.fit(X_train_w2v, y_train)
metrics_w2v = evaluate_and_log(
    y_val, clf_w2v.predict(X_val_w2v),
    model_name="Logistic Regression Baseline",
    feature_desc="Word2Vec Mean Pooling",
    params="vector_size=100, window=5, min_count=2, penalty=l2, class_weight=balanced"
)


MODEL EVALUATION: Logistic Regression Baseline (Word2Vec Mean Pooling)
[INFO] Accuracy          : 0.5018
[INFO] Precision (Macro) : 0.4432
[INFO] Recall (Macro)    : 0.4631
[INFO] F1 (Macro)        : 0.4339
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.20      0.35      0.26       288
     Bullish       0.35      0.50      0.41       385
     Neutral       0.78      0.54      0.64      1236

    accuracy                           0.50      1909
   macro avg       0.44      0.46      0.43      1909
weighted avg       0.61      0.50      0.53      1909

------------------------------------------------------------
[[101  96  91]
 [ 99 193  93]
 [302 270 664]]
[INFO] Updated run in outputs/results.csv


In [33]:
clf_glove = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000,
                                class_weight="balanced", random_state=SEED)
clf_glove.fit(X_train_glove, y_train)
metrics_glove = evaluate_and_log(
    y_val, clf_glove.predict(X_val_glove),
    model_name="Logistic Regression Baseline",
    feature_desc="GloVe-Twitter-100 Mean Pooling",
    params="vector_size=100, pre-trained, penalty=l2, class_weight=balanced"
)


MODEL EVALUATION: Logistic Regression Baseline (GloVe-Twitter-100 Mean Pooling)
[INFO] Accuracy          : 0.6590
[INFO] Precision (Macro) : 0.5735
[INFO] Recall (Macro)    : 0.6181
[INFO] F1 (Macro)        : 0.5855
------------------------------------------------------------
              precision    recall  f1-score   support

     Bearish       0.37      0.55      0.44       288
     Bullish       0.49      0.61      0.54       385
     Neutral       0.86      0.70      0.77      1236

    accuracy                           0.66      1909
   macro avg       0.57      0.62      0.59      1909
weighted avg       0.71      0.66      0.68      1909

------------------------------------------------------------
[[157  66  65]
 [ 76 234  75]
 [189 180 867]]
[INFO] Updated run in outputs/results.csv


## 9. Error Analysis

In [34]:
from src.error_analysis import run_error_analysis

# Refit champion model to obtain probabilities (run_model_pipeline does not return them)
clf_ea = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000,
                            class_weight="balanced", random_state=SEED)
clf_ea.fit(X_train_opt, y_train)
y_pred_ea  = clf_ea.predict(X_val_opt)
y_proba_ea = clf_ea.predict_proba(X_val_opt)

run_error_analysis(y_val, y_pred_ea, y_proba_ea, X_val_raw, X_val_pre)


------------------------------------------------------------
[INFO] Val F1 Macro : 0.7091
[INFO] Val Accuracy : 0.7795
------------------------------------------------------------
[SUCCESS] Confusion matrix saved to outputs/confusion_matrix.png
[INFO] Collecting misclassifications ...
[SUCCESS] TXT report saved to outputs/misclassified_report.txt
[SUCCESS] JSON report saved to outputs/misclassified_analysis.json


## 10. Generate Test Predictions (Best Classical Model)

Retrain the best classical model on 100% of training data, then run inference on the test set.

In [35]:
from src.config import TEST_CSV_PATH

# Identify best model from leaderboard
results_df = pd.read_csv(RESULTS_CSV_PATH)
best_row = results_df.sort_values("f1_macro", ascending=False).iloc[0]
print(f"Best model : {best_row['model_name']}")
print(f"Features   : {best_row['feature_description']}")
print(f"Params     : {best_row['parameters']}")
print(f"Val F1     : {best_row['f1_macro']}")


Best model : Logistic Regression Baseline
Features   : TF-IDF (1,2) Optimized
Params     : ngram_range=(1, 2), min_df=2, max_features=25000
Val F1     : 0.7113


In [36]:
# Preprocess & vectorize 100% of training data
train_full_df = pd.read_csv(TRAIN_CSV_PATH)
X_full_pre = train_full_df["text"].apply(lambda t: preprocess_tweet(t, return_str=True))
y_full = train_full_df["label"]

vec_final = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000)
X_full_vec = vec_final.fit_transform(X_full_pre)
print(f"Full training matrix: {X_full_vec.shape}")


Full training matrix: (9543, 13675)


In [37]:
# Fit best model on full training data
best_model = LogisticRegression(penalty="l2", solver="lbfgs", max_iter=1000,
                                 class_weight="balanced", random_state=SEED)
best_model.fit(X_full_vec, y_full)
print("Model fitted on 100% training data.")


Model fitted on 100% training data.


In [38]:
from src.evaluate import save_submission

test_df = pd.read_csv(TEST_CSV_PATH)
X_test_pre = test_df["text"].apply(lambda t: preprocess_tweet(t, return_str=True))
X_test_vec = vec_final.transform(X_test_pre)
y_pred_test = best_model.predict(X_test_vec)

submission = save_submission(test_df, y_pred_test)
submission.head()


[SUCCESS] Predictions saved to outputs/pred_best.csv (2388 rows)


,id,label
0,0,2
1,1,2
2,2,2
3,3,2
4,4,2


## 11. DistilBERT Spike (200-sample sanity check)

In [39]:
from src.distilbert_trainer import run_spike
run_spike()


[INFO] No CUDA device found — using CPU
[INFO] Loading tokenizer: distilbert-base-uncased


[INFO] Vocab size: 30522
[INFO] Loaded 200 samples from data/train.csv
[INFO] input_ids shape        : (200, 78)
[INFO] attention_mask shape   : (200, 78)
[INFO] Sample tokens (row 0)  : [101, 1002, 2011, 4859, 1011, 16545, 5302, 16998, 15934, 2015] ...
[SUCCESS] Spike complete — environment is ready for DistilBERT fine-tuning.


## 12. DistilBERT Full Fine-Tuning

> **Note:** Full training requires a GPU. Set `num_train_epochs` and `per_device_train_batch_size` according to available hardware.
> Expected time: ~15–30 min on a single GPU.

In [40]:
from src.distilbert_trainer import run_trainer
run_trainer(n_samples=None)


[INFO] Loading all samples ...
[INFO] Split — train=7634, val=1909
[INFO] Loading tokenizer: distilbert-base-uncased
[INFO] Vocab size: 30522
[INFO] Loading dataset from cache: outputs\distilbert_cache\train
[INFO] Loading dataset from cache: outputs\distilbert_cache\val
[INFO] Datasets — train=7634 rows, val=1909 rows
[INFO] Loading DistilBertForSequenceClassification ...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[INFO] Starting training ...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.468688,0.428509,0.844945,0.786555
2,0.278621,0.410289,0.860660,0.810159
3,0.183095,0.433961,0.865375,0.821044


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[INFO] Running final evaluation ...


[INFO] Updated run in outputs/results.csv


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro
0.183095,0.433961,3,0.865375,0.821044


[SUCCESS] Training complete.


## 13. Full Leaderboard (All Experiments)

In [41]:
results_df = pd.read_csv(RESULTS_CSV_PATH)
print(f"Total logged runs: {len(results_df)}")
display(
    results_df
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
    .style.background_gradient(cmap="YlGn", subset=["f1_macro"])
    .format("{:.4f}", subset=["accuracy", "precision_macro", "recall_macro", "f1_macro"])
)


Total logged runs: 43


,timestamp,owner,model_name,feature_description,accuracy,precision_macro,recall_macro,f1_macro,parameters,f1_per_class,notes
0,2026-05-25 02:06:17,nan,DistilBERT,HF fine-tune,0.8654,0.8325,0.8108,0.8210,"model=distilbert-base-uncased, max_length=128","{'Bearish': 0.7602131438721137, 'Bullish': 0.7924528301886793, 'Neutral': 0.9104655789892558}",nan
1,2026-05-23 17:46:33,Filipe,Logistic Regression Baseline,"TF-IDF (1,2) Optimized",0.7810,0.7041,0.7201,0.7113,"penalty=l2, solver=lbfgs, C=1.0, class_weight=balanced",nan,nan
2,2026-05-23 17:46:25,Filipe,Logistic Regression Baseline,"TF-IDF (1,2) Optimized",0.7810,0.7041,0.7201,0.7113,"ngram_range=(1, 2), min_df=2, max_features=25000",nan,nan
3,2026-05-25 01:20:32,nan,Logistic Regression Baseline,"TF-IDF (1,2) Optimized",0.7795,0.7021,0.7175,0.7091,"penalty=l2, solver=lbfgs, C=1.0, class_weight=balanced","{'Bearish': 0.5980392156862745, 'Bullish': 0.669260700389105, 'Neutral': 0.859958932238193}",nan
4,2026-05-25 01:20:25,nan,Logistic Regression Baseline,"TF-IDF (1,2) Optimized",0.7795,0.7021,0.7175,0.7091,"ngram_range=(1, 2), min_df=2, max_features=25000","{'Bearish': 0.5980392156862745, 'Bullish': 0.669260700389105, 'Neutral': 0.859958932238193}",nan
5,2026-05-25 01:21:13,nan,MLP Classifier,"TF-IDF (1,2) Optimized",0.8025,0.7590,0.6728,0.7051,"hidden_layer_sizes=(100, 50), learning_rate_init=0.001, early_stopping=True","{'Bearish': 0.5585215605749486, 'Bullish': 0.6763425253991292, 'Neutral': 0.8803936411809236}",nan
6,2026-05-25 01:20:25,nan,Logistic Regression Baseline,"TF-IDF (1,2)",0.7800,0.7019,0.7057,0.7038,"ngram_range=(1, 2), min_df=1, max_features=None","{'Bearish': 0.5862068965517241, 'Bullish': 0.6623711340206185, 'Neutral': 0.8627132412672623}",nan
7,2026-05-23 17:23:25,Filipe,Logistic Regression Baseline,"TF-IDF (1,2)",0.7795,0.7015,0.7048,0.7031,"max_iter=1000, class_weight='balanced', random_state=42",nan,nan
8,2026-05-23 17:46:25,Filipe,Logistic Regression Baseline,"TF-IDF (1,2)",0.7795,0.7015,0.7048,0.7031,"ngram_range=(1, 2), min_df=1, max_features=None",nan,nan
9,2026-05-23 17:22:40,Filipe,Logistic Regression Baseline,"TF-IDF (1,2)",0.7795,0.7015,0.7048,0.7031,"max_iter=1000, class_weight='balanced', random_state=42",nan,nan
